In [22]:
import json

with open("../data/json_files/support_tickets.json", encoding="utf-8") as f:
    json_data = json.load(f)

In [23]:
json_data[0]

{'ticket_id': 501,
 'client': 'TechVision',
 'priority': 'high',
 'category': 'API',
 'agent': 'Alice Dupont',
 'details': {'summary': 'Problème de connexion à l’API',
  'steps': ['Test des identifiants',
   'Analyse des logs',
   'Escalade au support L2'],
  'created_at': '2024-01-15T09:14:00Z'},
 'tags': ['API', 'incident', 'urgent'],
 'comments': [{'from': 'client',
   'text': 'Impossible de se connecter depuis ce matin'},
  {'from': 'agent', 'text': "Ticket en cours d'analyse"}]}

## Fonction récursive  json_to_text()

In [24]:
def json_to_text(obj, prefix = ""):
        lines = []
        if isinstance(obj, dict):
            for k, v in obj.items():
                lines.append(json_to_text(v, prefix=f"{prefix}{k}."))
        elif isinstance(obj, list):
            for i, item in enumerate(obj):
                lines.append(json_to_text(item, prefix=f"{prefix}{i}."))
        else:
                lines.append(f"{prefix[:-1]} : {obj}")
        return "\n".join(lines)


In [25]:
from langchain_core.documents import Document
docs_json = [
    Document(
        page_content=json_to_text(ticket),
        metadata={"ticket_id": ticket.get("ticket_id")},
    )
    for ticket in json_data
]


In [26]:
docs_json

[Document(metadata={'ticket_id': 501}, page_content="ticket_id : 501\nclient : TechVision\npriority : high\ncategory : API\nagent : Alice Dupont\ndetails.summary : Problème de connexion à l’API\ndetails.steps.0 : Test des identifiants\ndetails.steps.1 : Analyse des logs\ndetails.steps.2 : Escalade au support L2\ndetails.created_at : 2024-01-15T09:14:00Z\ntags.0 : API\ntags.1 : incident\ntags.2 : urgent\ncomments.0.from : client\ncomments.0.text : Impossible de se connecter depuis ce matin\ncomments.1.from : agent\ncomments.1.text : Ticket en cours d'analyse"),
 Document(metadata={'ticket_id': 502}, page_content='ticket_id : 502\nclient : NovaBank\npriority : critical\ncategory : Sécurité\nagent : Mehmet Kaya\ndetails.summary : Tentatives de connexion suspectes\ndetails.steps.0 : Blocage temporaire du compte\ndetails.steps.1 : Analyse comportementale\ndetails.steps.2 : Transmission à l’équipe sécurité\ndetails.created_at : 2024-01-16T11:40:00Z\ntags.0 : security\ntags.1 : fraud\ncomment

In [27]:
for i, doc in enumerate(docs_json[:2]):
        print("="*70)
        print("DOCUMENT JSON", i)
        print("="*70)
        print(doc.page_content)
        print()

DOCUMENT JSON 0
ticket_id : 501
client : TechVision
priority : high
category : API
agent : Alice Dupont
details.summary : Problème de connexion à l’API
details.steps.0 : Test des identifiants
details.steps.1 : Analyse des logs
details.steps.2 : Escalade au support L2
details.created_at : 2024-01-15T09:14:00Z
tags.0 : API
tags.1 : incident
tags.2 : urgent
comments.0.from : client
comments.0.text : Impossible de se connecter depuis ce matin
comments.1.from : agent
comments.1.text : Ticket en cours d'analyse

DOCUMENT JSON 1
ticket_id : 502
client : NovaBank
priority : critical
category : Sécurité
agent : Mehmet Kaya
details.summary : Tentatives de connexion suspectes
details.steps.0 : Blocage temporaire du compte
details.steps.1 : Analyse comportementale
details.steps.2 : Transmission à l’équipe sécurité
details.created_at : 2024-01-16T11:40:00Z
tags.0 : security
tags.1 : fraud
comments.0.from : client
comments.0.text : On m’a signalé un accès non autorisé
comments.1.from : agent
comment

In [29]:
# chunker les documents nettoyés
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=80,
)
final_chunks = splitter.split_documents(docs_json)

print(f"Nombre de chunks : {len(final_chunks)}")

Nombre de chunks : 3
